# Option A v4 — generated-image integration verification ONLY
Choose **CPU / no accelerator**, then run cells in order in a fresh Colab runtime.
Upload ONLY `vision_option_a_export_code_v4.zip`. No Drive mount, dental images, dataset manifest, owner authorization or weights are needed.
This notebook has no training/pilot call and no installation commands. Generated source fixtures use synthetic permissions that cannot authorize real packaging.
Expected selected checks: **131**, with zero failures/errors/skips. Earlier QA/isolated/model workflows are not rerun or included in that count.
The new v4 data bundle is deliberately rejected by the historical v3 pilot reader. This notebook verifies the exporter only; it does not activate a real-data consumer.


In [ ]:
import sys, platform, importlib.util
from pathlib import Path
RUN_PILOT = False
assert RUN_PILOT is False
for name in ("numpy", "PIL", "pytest"):
    if importlib.util.find_spec(name) is None:
        raise RuntimeError("Missing dependency: " + name + "; stop, do not auto-install")
import numpy, PIL, pytest
from PIL import ImageCms, features
if not features.check("littlecms2"):
    raise RuntimeError("Pillow LittleCMS unavailable; stop")
VERSIONS = {"python": platform.python_version(), "numpy": numpy.__version__,
            "pillow": PIL.__version__, "pytest": pytest.__version__,
            "littlecms": features.version("littlecms2"), "execution": "CPU generated-only"}
print(VERSIONS)


In [ ]:
from google.colab import files
import hashlib, json, zipfile, stat, io
EXPECTED_ZIP_NAME = "vision_option_a_export_code_v4.zip"
EXPECTED_ZIP_SHA = '1c512d1fc29c9c1b3eae55750a89a47e8ce5ad09a4fad9042c4676fff8e3e00e'
EXPECTED_FILES = {'src/vision/exploratory_bundle.py': 'dda776ca1558ed4da795df7cee47c0fe19d382cf32c37ca89d22f4141974b615', 'src/vision/exploratory_bundle_v4.py': 'c914cc05ad2be4ff08873a5d2270b5836e0f84219a539435beeca09b57ad76cd', 'src/vision/colour_qa.py': '4ad2b81c715b01e624b527d1971cc303740ddf559461f2d953c831956c1e82e3', 'src/vision/colour_policy.py': '2bba567545143687f49b6e3ae63fdd72d1eacc10977d5ec373f00a4b5451a97b', 'src/vision/model_pipeline.py': 'c11ad18912575f3567d0baf0eefb1bfbb5eb00fbfa075880fe84aef1ccd34b3c', 'src/vision/train_pipeline.py': 'e7a979b0875af0e459496bca79ce3c80e594acf2639ea7dc9d8659ec75ffc695', 'tests/test_exploratory_bundle_v4.py': 'd73f0ab8c3edadb525ac876f6f78ff9713d57d2b9cc5aa146ead0dde2643889e', 'tests/test_exploratory_bundle.py': '2729fbbd8a882a0fba2af2546c5c91c14833fe6a15376ee0ee487168a350ac46', 'tests/test_icc_colour_export.py': '599e10096500a4ee3daafff88d00ca856ebffa4e6e4b89c4c80c21713c07772f', 'tests/test_colour_qa.py': '4f81b592be123fa746fa2c7ace92a6e9e8249e1cbcbd29a8b3a6c0d9ec0510a1', 'tests/test_colour_policy.py': '0b3b2b16d0666b28cceb5044698cb6d7638f31532a2ba1f6ab43e7624da5e5f9'}
uploaded = files.upload()
if set(uploaded) != {EXPECTED_ZIP_NAME}:
    raise RuntimeError("Upload only the named code-only ZIP; stop")
raw = uploaded[EXPECTED_ZIP_NAME]
if len(raw) > 8*1024**2 or hashlib.sha256(raw).hexdigest() != EXPECTED_ZIP_SHA:
    raise RuntimeError("Code archive hash/size mismatch")
WORK = Path("/content/vision_option_a_v4_generated_verification")
if WORK.exists():
    raise RuntimeError("Use a fresh runtime; no existing files will be overwritten")
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    entries = z.infolist()
    if (len(entries) != len(EXPECTED_FILES)+1
            or set(z.namelist()) != set(EXPECTED_FILES) | {"code_manifest.json"}
            or any(e.file_size > 2*1024**2 or stat.S_ISLNK(e.external_attr >> 16) for e in entries)
            or sum(e.file_size for e in entries) > 8*1024**2 or z.testzip() is not None):
        raise RuntimeError("Unsafe code archive")
    manifest = json.loads(z.read("code_manifest.json"))
    if manifest["files"] != EXPECTED_FILES:
        raise RuntimeError("Source manifest mismatch")
    payloads = {name: z.read(name) for name in EXPECTED_FILES}
    for name, content in payloads.items():
        if hashlib.sha256(content).hexdigest() != EXPECTED_FILES[name]:
            raise RuntimeError("Source hash mismatch: " + name)
        compile(content, name, "exec")
    WORK.mkdir()
    for name, content in payloads.items():
        target = WORK / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
    (WORK / "code_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Verified", len(EXPECTED_FILES), "code/test SHA-256 hashes; no dataset included")


In [ ]:
import subprocess, os, xml.etree.ElementTree as ET
TARGETS = ["tests/test_exploratory_bundle_v4.py", "tests/test_exploratory_bundle.py", "tests/test_icc_colour_export.py"]
EVIDENCE = WORK / "synthetic_evidence"
EVIDENCE.mkdir()
def source_hashes():
    return {name: hashlib.sha256((WORK/name).read_bytes()).hexdigest() for name in EXPECTED_FILES}
before = source_hashes()
assert before == EXPECTED_FILES
command = [sys.executable, "-B", "-m", "pytest", *TARGETS, "-q", "-p", "no:cacheprovider",
           "--junitxml=" + str(EVIDENCE/"junit.xml")]
env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1", PYTHONDONTWRITEBYTECODE="1", PYTHONPATH=str(WORK))
run = subprocess.run(command, cwd=WORK, env=env, text=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, timeout=180)
(EVIDENCE/"pytest.log").write_text(run.stdout)
print(run.stdout)
after = source_hashes()
cases = list(ET.parse(EVIDENCE/"junit.xml").iter("testcase")) if (EVIDENCE/"junit.xml").exists() else []
failed = sum(c.find("failure") is not None for c in cases)
errors = sum(c.find("error") is not None for c in cases)
skipped = sum(c.find("skipped") is not None for c in cases)
result = {"versions": VERSIONS, "zip_sha256": EXPECTED_ZIP_SHA, "source_before": before,
          "source_after": after, "source_unchanged": before == after == EXPECTED_FILES,
          "test_targets": TARGETS, "total": len(cases), "passed": len(cases)-failed-errors-skipped,
          "failed": failed, "errors": errors, "skipped": skipped, "exit_code": run.returncode,
          "RUN_PILOT": RUN_PILOT, "real_images_accessed": False,
          "scope": "Selected generated-image export integration only; not clinical/model validation"}
(EVIDENCE/"verification.json").write_text(json.dumps(result, indent=2)+"\n")
print({key: result[key] for key in ("total", "passed", "failed", "errors", "skipped", "source_unchanged")})


In [ ]:
OUTPUT = Path("/content/vision_option_a_export_runtime_evidence_v4.zip")
with zipfile.ZipFile(OUTPUT, "x", compression=zipfile.ZIP_DEFLATED) as z:
    for name in ("pytest.log", "junit.xml", "verification.json"):
        if (EVIDENCE/name).exists(): z.write(EVIDENCE/name, name)
files.download(str(OUTPUT))
assert (run.returncode == 0 and result["passed"] == 131 and result["failed"] == 0
        and result["errors"] == 0 and result["skipped"] == 0 and result["source_unchanged"])
print("PASS: selected generated integration checks only. No real packaging/training authorized.")


Download the evidence even when tests fail, then report the exact failure. Do not install a new library, change policy limits, use a fallback, upload images or run the pilot to force a pass.
After success, request a **separate local-packaging-only owner decision** with the current v4 code/policy/DRAFT/plan hashes and a new output ID. Current source feasibility and actual data-use requirements must still pass. Upload, a v4-compatible pilot consumer and dental training require separate scoped work/permission. No clinical label, participant independence or final-research approval is created here.
